In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix)

In [2]:
CHART_DIR = "charts"
os.makedirs(CHART_DIR, exist_ok=True)
sns.set_style("whitegrid")
df = pd.read_csv("delhi_business_clean.csv")
df["price_level_clean"] = df["price_level_clean"].fillna(0)
print(f"\nBusinesses WITH a website   : {(df['has_website']==1).sum()}")
print(f"Businesses WITHOUT a website: {(df['has_website']==0).sum()}")


Businesses WITH a website   : 6004
Businesses WITHOUT a website: 5780


### FEATURES AND TARGETS

In [10]:
FEATURES_NUM = ["review_count", "rating", "num_types", "price_level_clean"]
FEATURES_CAT = ["area", "category_searched"]
TARGET = "has_website"
X = df[FEATURES_NUM + FEATURES_CAT]
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"\nTrain size: {len(X_train)}, Test size: {len(X_test)}")

preprocessor = ColumnTransformer(transformers=[("cat", OneHotEncoder(handle_unknown="ignore"), FEATURES_CAT),("num", StandardScaler(), FEATURES_NUM),])                         


Train size: 9427, Test size: 2357


### MODEL 1: LOGISTIC REGRESSION (baseline)

In [11]:
log_pipeline = Pipeline([("prep", preprocessor),("model", LogisticRegression(max_iter=1000, class_weight="balanced"))])
log_pipeline.fit(X_train, y_train)
log_pred = log_pipeline.predict(X_test)
log_proba = log_pipeline.predict_proba(X_test)[:, 1]

### MODEL 2: RANDOM FOREST

In [12]:
rf_pipeline = Pipeline([("prep", preprocessor),("model", RandomForestClassifier(n_estimators=300, max_depth=12,random_state=42, n_jobs=-1, class_weight="balanced"))])
rf_pipeline.fit(X_train, y_train)
rf_pred = rf_pipeline.predict(X_test)
rf_proba = rf_pipeline.predict_proba(X_test)[:, 1]

### EVALUATION OF THE MODELS

In [ ]:
def evaluate(name, y_true, y_pred, y_proba):
    return {
        "Model": name,
        "Accuracy": round(accuracy_score(y_true, y_pred), 4),
        "Precision": round(precision_score(y_true, y_pred), 4),
        "Recall": round(recall_score(y_true, y_pred), 4),
        "F1": round(f1_score(y_true, y_pred), 4),
        "ROC_AUC": round(roc_auc_score(y_true, y_proba), 4),
    }
results = pd.DataFrame([evaluate("Logistic Regression", y_test, log_pred, log_proba), evaluate("Random Forest", y_test, rf_pred, rf_proba),])
print("\nMODEL COMPARISON:")
print(results.to_string(index=False))
results.to_csv("website_propensity_model_results.csv", index=False)
print("\n✅ Saved website_propensity_model_results.csv")
best_model_name = results.loc[results["ROC_AUC"].idxmax(), "Model"]
print(f"\n🏆 Best model by ROC-AUC: {best_model_name}")


MODEL COMPARISON:
              Model  Accuracy  Precision  Recall     F1  ROC_AUC
Logistic Regression    0.6606     0.6795  0.6320 0.6549   0.7268
      Random Forest    0.7043     0.7165  0.6944 0.7053   0.7873

✅ Saved website_propensity_model_results.csv

🏆 Best model by ROC-AUC: Random Forest


### FEATURE IMPORTANCE

In [ ]:
feature_names = rf_pipeline.named_steps["prep"].get_feature_names_out()
importances = rf_pipeline.named_steps["model"].feature_importances_
imp_df = pd.DataFrame({"feature": feature_names, "importance": importances})
imp_df = imp_df.sort_values("importance", ascending=False).head(15)
plt.figure(figsize=(10, 7))
sns.barplot(data=imp_df, x="importance", y="feature", palette="mako")
plt.title("What Predicts a Business Having a Website? (Feature Importance)",fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CHART_DIR}/07_feature_importance.png")
plt.close()

C:\Users\georg\AppData\Local\Temp\ipykernel_16688\2264111595.py:7: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=imp_df, x="importance", y="feature", palette="mako")


### ROC CURVE + CONFUSION MATRIX

In [15]:
fpr, tpr, _ = roc_curve(y_test, rf_proba)
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
axes[0].plot(fpr, tpr, color="teal", label=f"Random Forest (AUC={roc_auc_score(y_test, rf_proba):.2f})")
axes[0].plot([0, 1], [0, 1], "r--", label="Random guess")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curve")
axes[0].legend()
cm = confusion_matrix(y_test, rf_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[1], xticklabels=["No Website", "Has Website"], yticklabels=["No Website", "Has Website"])
axes[1].set_title("Confusion Matrix")
axes[1].set_ylabel("Actual")
axes[1].set_xlabel("Predicted")

plt.tight_layout()
plt.savefig(f"{CHART_DIR}/08_roc_and_confusion.png")
plt.close()

### SCORING EVERY BUSINESS

In [16]:
df["website_propensity_score"] = rf_pipeline.predict_proba(X)[:, 1]
df.to_csv("delhi_business_scored.csv", index=False)

WHAT THIS MODEL DOES:

It learns what a business "with a website" typically looks like
(category, area, rating, review_count, price signals), then scores every
business WITHOUT a website on how closely it matches that profile.
 
A business that scores high but has no website is a strong lead: it's
successful/established enough to look just like businesses that already
invested in a web presence -- it's simply missing one. That's the gap a
web developer can pitch into.